In [0]:
taxi_df = spark.table("week2_silver_taxi_time_features")
zones_df = spark.table("week1_real_bronze_taxi_zones")

display(taxi_df.limit(10))
taxi_df.printSchema()

display(zones_df.limit(10))
zones_df.printSchema()

In [0]:
from pyspark.sql.functions import col, count as spark_count, when

In [0]:
zones_clean_df = zones_df.select(
    col("LocationID").cast("int").alias("location_id"),
    col("Borough").alias("borough"),
    col("Zone").alias("zone_name"),
    col("service_zone").alias("service_zone")
)

display(zones_clean_df.limit(20))
zones_clean_df.printSchema()

In [0]:
duplicate_zone_ids_df = (
    zones_clean_df
    .groupBy("location_id")
    .agg(spark_count("*").alias("record_count"))
    .filter(col("record_count") > 1)
)

display(duplicate_zone_ids_df)

In [0]:
taxi_pickup_enriched_df = (
    taxi_df.alias("t")
    .join(
        zones_clean_df.alias("z"),
        col("t.pickup_location_id") == col("z.location_id"),
        "left"
    )
    .select(
        col("t.*"),
        col("z.borough").alias("pickup_borough"),
        col("z.zone_name").alias("pickup_zone"),
        col("z.service_zone").alias("pickup_service_zone")
    )
)

display(taxi_pickup_enriched_df.limit(20))

In [0]:
taxi_full_enriched_df = (
    taxi_pickup_enriched_df.alias("t")
    .join(
        zones_clean_df.alias("z"),
        col("t.dropoff_location_id") == col("z.location_id"),
        "left"
    )
    .select(
        col("t.*"),
        col("z.borough").alias("dropoff_borough"),
        col("z.zone_name").alias("dropoff_zone"),
        col("z.service_zone").alias("dropoff_service_zone")
    )
)

display(taxi_full_enriched_df.limit(20))

In [0]:
missing_pickup_zones = taxi_full_enriched_df.filter(col("pickup_zone").isNull()).count()
missing_dropoff_zones = taxi_full_enriched_df.filter(col("dropoff_zone").isNull()).count()

print(f"Missing pickup zones: {missing_pickup_zones}")
print(f"Missing drop-off zones: {missing_dropoff_zones}")

In [0]:
taxi_full_enriched_df = taxi_full_enriched_df.withColumn(
    "borough_trip_type",
    when(col("pickup_borough") == col("dropoff_borough"), "Within same borough")
    .otherwise("Between boroughs")
)

display(taxi_full_enriched_df.select(
    "pickup_borough",
    "dropoff_borough",
    "borough_trip_type"
).limit(50))

In [0]:
taxi_full_enriched_df.write.mode("overwrite").saveAsTable("week2_silver_taxi_enriched")

In [0]:
display(spark.sql("SHOW TABLES LIKE 'week2_silver_taxi_enriched'"))

In [0]:
saved_enriched_df = spark.table("week2_silver_taxi_enriched")

display(saved_enriched_df.limit(10))
saved_enriched_df.printSchema()

In [0]:
base_count = spark.table("week2_silver_taxi_time_features").count()
enriched_count = spark.table("week2_silver_taxi_enriched").count()

print(f"Base Day 2 table rows: {base_count}")
print(f"Enriched Day 3 table rows: {enriched_count}")
print(f"Row count difference: {enriched_count - base_count}")

In [0]:
display(
    spark.table("week2_silver_taxi_enriched")
    .groupBy("borough_trip_type")
    .count()
)